# M7.4 · Dağıtık eğitim: DDP, all-reduce, ZeRO/FSDP

Birden çok işçiyle eğitmek. İki işçinin all-reduce'unu, ring all-reduce'un iletişim hacmini ve ZeRO-3 bellek payını elle hesaplıyor, saf Python'da ring all-reduce'u adım adım simüle ediyor, NumPy'da veri paralel SGD'nin tam batch gradyanına eşit olduğunu ve optimizer durumunu işçilere bölen ZeRO-1 Adam'ın tek işçili Adam'la aynı sonucu verdiğini gösteriyor, PyTorch'ta `gloo` ile iki CPU sürecinde DDP eğitiyor ve en sonda gerçek kolektiflerle (all-reduce + all-gather) kendi ZeRO-1 optimizer'ını yazıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m7-4-ddp-fsdp.html](https://m-gonul.github.io/pytorch-lab/m7-4-ddp-fsdp.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m7_4_ddp_fsdp.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    rng = random.Random(0)\n\n    def t(N, c):\n        def f():\n            vecs = [[rng.randint(-9, 9) for _ in range(N * c)] for _ in range(N)]\n            total = [sum(v[j] for v in vecs) for j in range(N * c)]\n            res, sent = need(ns, \"ring_allreduce\")([list(v) for v in vecs])\n            for i in range(N):\n                ensure(list(res[i]) == total, f\"işçi {i} toplamı yanlış\")\n            ensure(list(sent) == [2 * (N - 1) * c] * N, f\"gönderilen {sent}, beklenen her işçi {2 * (N - 1) * c}\")\n            return f\"N = {N}: işçi başına {2 * (N - 1) * c} / {N * c} eleman = {2 * (N - 1) / N:.2f}×\"\n        return f\n    R.test(\"N = 2\", t(2, 3))\n    R.test(\"N = 4\", t(4, 2))\n    R.test(\"N = 8\", t(8, 5))\n\n    def orig():\n        vecs = [[1, 2], [3, 4]]\n        need(ns, \"ring_allreduce\")(vecs)\n    R.test(\"girdi değişmese de olur (kopya)\", orig)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"adam_update döngüsüz\", lambda: max_loops(src, 0, func=\"adam_update\"))\n    R.test(\"zero1_step: yalnız işçi döngüsü\", lambda: max_loops(src, 1, func=\"zero1_step\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    X, y, w = rng.normal(size=(64, 6)), rng.normal(size=64), rng.normal(size=6)\n\n    def dp():\n        for K in (1, 2, 4, 8):\n            gs = need(ns, \"worker_grads\")(w, X, y, K)\n            ensure(len(gs) == K, f\"K = {K} gradyan\")\n            close(need(ns, \"allreduce_mean\")(gs), 2 * X.T @ (X @ w - y) / 64, what=f\"K = {K}\")\n    R.test(\"veri paralel gradyan = tam batch gradyanı\", dp)\n\n    def ad():\n        g = rng.normal(size=6); m = rng.normal(size=6) * 0.1; v = rng.random(6) * 0.1\n        w2, m2, v2 = need(ns, \"adam_update\")(w, g, m, v, 3)\n        mm = 0.9 * m + 0.1 * g; vv = 0.999 * v + 0.001 * g * g\n        close(w2, w - 1e-2 * (mm / (1 - 0.9 ** 3)) / (np.sqrt(vv / (1 - 0.999 ** 3)) + 1e-8))\n    R.test(\"adam_update (bias düzeltmeli)\", ad)\n\n    def z1():\n        W = rng.normal(size=10)\n        Wf, m, v = W.copy(), np.zeros(10), np.zeros(10)\n        K = 3\n        states = [(np.zeros(len(s)), np.zeros(len(s))) for s in np.array_split(W, K)]\n        Wz = W.copy()\n        for t in range(1, 21):\n            g = np.sin(np.arange(10) * t) + Wz * 0.1\n            gf = np.sin(np.arange(10) * t) + Wf * 0.1\n            Wf, m, v = need(ns, \"adam_update\")(Wf, gf, m, v, t)\n            Wz, states = need(ns, \"zero1_step\")(Wz, g, states, t)\n        close(Wz, Wf)\n        sizes = [len(s[0]) for s in states]\n        ensure(sizes == [4, 3, 3], f\"işçi başına durum boyutu {sizes}\")\n        return f\"20 adım, 3 işçi: aynı ağırlıklar; durum boyutları {sizes} (toplam 10)\"\n    R.test(\"ZeRO-1 = tek işçili Adam\", z1)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef reference(ns, steps):\n    torch.manual_seed(0)\n    m = ns[\"make_model\"]()\n    X, y = ns[\"make_data\"]()\n    opt = torch.optim.SGD(m.parameters(), lr=0.1)\n    for _ in range(steps):\n        loss = ((m(X) - y) ** 2).mean()\n        opt.zero_grad(); loss.backward(); opt.step()\n    return [p.detach() for p in m.parameters()]\n\n\ndef check(ns, R):\n    need(ns, \"make_model\"); need(ns, \"make_data\")\n\n    def ddp():\n        P = need(ns, \"run\")(2, 5, False)\n        ensure(len(P) == 2, \"iki işçinin parametreleri\")\n        for a, b in zip(P[0], P[1]):\n            close(a, b, atol=0, rtol=0, what=\"iki işçi birebir aynı\")\n        for a, b in zip(P[0], reference(ns, 5)):\n            close(a, b, atol=1e-6, what=\"tek süreçli tam batch ile\")\n    R.test(\"DDP (2 süreç) = tek süreçli tam batch\", ddp)\n\n    def manual():\n        P = need(ns, \"run\")(2, 5, True)\n        for a, b in zip(P[0], reference(ns, 5)):\n            close(a, b, atol=1e-6)\n    R.test(\"DDP'siz, elle all-reduce\", manual)\n\n    def four():\n        P = need(ns, \"run\")(4, 3, True)\n        for r in range(1, 4):\n            for a, b in zip(P[0], P[r]):\n                close(a, b, atol=0, rtol=0, what=f\"işçi {r}\")\n        for a, b in zip(P[0], reference(ns, 3)):\n            close(a, b, atol=1e-6)\n        return \"4 süreç, 16'şar örnek: tek süreçli 64'lük batch ile aynı\"\n    R.test(\"4 süreç\", four)\n",
 "e": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef reference(ns, steps, lr=1e-2):\n    m = ns[\"make_model\"]()\n    X, y = ns[\"make_data\"]()\n    opt = torch.optim.Adam(m.parameters(), lr=lr)\n    for _ in range(steps):\n        loss = ((m(X) - y) ** 2).mean()\n        opt.zero_grad(); loss.backward(); opt.step()\n    return [p.detach() for p in m.parameters()], sum(p.numel() for p in m.parameters())\n\n\ndef check(ns, R):\n    def fl():\n        ts = [torch.randn(3, 4), torch.randn(5), torch.randn(2, 1, 2)]\n        f = need(ns, \"flatten\")(ts)\n        ensure(f.shape == (21,), \"düz vektör 21 eleman\")\n        for a, b in zip(need(ns, \"unflatten\")(f, ts), ts):\n            close(a, b)\n    R.test(\"flatten / unflatten\", fl)\n\n    def z(world):\n        def f():\n            P, sizes = need(ns, \"run_zero1\")(world, 10)\n            ref, n = reference(ns, 10)\n            for r in range(world):\n                for a, b in zip(P[r], ref):\n                    close(a, b, atol=1e-5, what=f\"işçi {r}\")\n            per = -(-n // world) * 2\n            ensure(all(s == per for s in sizes), f\"işçi başına durum {sizes}; {per} bekleniyordu (2 × ⌈{n}/{world}⌉)\")\n            R.table(f\"ZeRO-1, {world} işçi\", [\"\", \"eleman\"], [[\"parametre\", str(n)], [\"Adam durumu, tek işçi\", str(2 * n)], [\"Adam durumu, işçi başına\", str(per)]])\n        return f\n    R.test(\"ZeRO-1 (2 işçi) = torch.optim.Adam\", z(2))\n    R.test(\"ZeRO-1 (4 işçi; 1377 parametre 4'e bölünmüyor → dolgu)\", z(4))\n"
}

In [ ]:
import importlib, labcheck, checks_m7_4_ddp_fsdp as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Veri paralelliği:** her işçi (GPU/süreç) modelin tam bir kopyasını tutar, batch'in farklı bir parçasını işler. Gradyanlar işçiler arasında **ortalanır** (all-reduce); herkes aynı güncellemeyi yapar ve kopyalar senkron kalır. Ortalama kayıpla bu, tüm parçaların birleşimi üzerindeki tek bir büyük batch adımına **eşittir**.

**Ring all-reduce:** vektörü $N$ parçaya böl; işçiler halka hâlinde komşularına parça gönderir.

1. *Reduce-scatter* ($N-1$ adım): her adımda alınan parça kendi parçana eklenir; sonunda her işçi bir parçanın tam toplamına sahip.
2. *All-gather* ($N-1$ adım): toplanmış parçalar halka boyunca dağıtılır.

İşçi başına gönderilen: $2(N-1)/N\cdot$ boyut.

**Bellek:** DDP'de her işçi parametre + gradyan + optimizer durumunun tamamını tutar. **ZeRO** bunları böler: ZeRO-1 optimizer durumunu, ZeRO-2 + gradyanları, ZeRO-3 (= FSDP) + parametreleri. ZeRO-1'de her işçi yalnız kendi parametre parçasını günceller, sonra parçalar all-gather ile birleştirilir.

`torch.distributed` CPU'da `gloo` arka ucuyla çalışır: bu modülde iki süreç aynı makinede `127.0.0.1` üzerinden konuşur.

## 1 · Elle

ZeRO-1'de parametre (4) ve gradyan (4) her GPU'da tam; yalnız Adam'ın iki momenti (8 bayt) 8'e bölünüyor: $4+4+1=9$ bayt/parametre. ZeRO-3'te her şey bölünür: $16/8=2$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**İki işçi, gradyanlar $g_1=(1,3)$, $g_2=(3,5)$**

- `ar`: All-reduce (ortalama) sonrası 1. bileşen
- `batch`: İşçi başına batch 32, 4 işçi: eşdeğer tek batch boyutu

**Ring all-reduce, $N=4$ işçi, 100 MB gradyan**

- `ring`: İşçi başına gönderilen veri (MB): $2(N-1)/N\cdot100$
- `steps`: Toplam adım sayısı (reduce-scatter + all-gather)

**1 milyar parametre, fp32 Adam (16 bayt/parametre), 8 GPU**

- `ddp`: DDP: GPU başına bayt
- `z3`: ZeRO-3 / FSDP (her şey bölünmüş): GPU başına bayt
- `z1`: ZeRO-1 (yalnız Adam durumu bölünmüş): GPU başına bayt

In [ ]:
cevaplar = dict(
    ar=None,
    batch=None,
    ring=None,
    steps=None,
    ddp=None,
    z3=None,
    z1=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"ar":[2,0],"batch":[128,0],"ring":[150,0],"steps":[6,0],"ddp":[16000000000,0],"z3":[2000000000,0],"z1":[9000000000,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, ring all-reduce simülasyonu. `vecs`: $N$ işçinin eşit uzunluklu listeleri (uzunluk $N$'ye bölünür).

- `ring_allreduce(vecs)` → `(sonuçlar, gönderilen)`: `sonuçlar[i]` işçi $i$'nin son vektörü (hepsi eleman toplamı), `gönderilen[i]` işçi $i$'nin gönderdiği eleman sayısı.
  - Reduce-scatter, adım $s=0..N-2$: işçi $i$, parça $(i-s)\bmod N$'yi işçi $(i+1)\bmod N$'ye gönderir; alıcı onu kendi aynı parçasına **ekler**.
  - All-gather, adım $s=0..N-2$: işçi $i$, parça $(i+1-s)\bmod N$'yi gönderir; alıcı **üzerine yazar**.
  - Her adımda tüm mesajları önce hazırla, sonra uygula (eşzamanlı gönderim).

In [ ]:
%%writefile stage_b.py
def ring_allreduce(vecs):
    ...


res, sent = ring_allreduce([[1, 3, 0, 0], [3, 5, 0, 0], [0, 0, 1, 1], [0, 0, 1, 1]])
print(res[0], sent)    # [4, 8, 2, 2], her işçi 2·3·1 = 6 eleman

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy simülasyonu, doğrusal regresyon $\ell=\frac1n\lVert Xw-y\rVert^2$.

- `worker_grads(w, X, y, K)` → $K$ işçinin kendi (eşit) parçalarındaki gradyan listesi.
- `allreduce_mean(grads)` → ortalama.
- `adam_update(w, g, m, v, t, lr=1e-2, b1=0.9, b2=0.999, eps=1e-8)` → `(w, m, v)`; bias düzeltmeli. Döngü yok.
- `zero1_step(w, g, states, t, lr=1e-2)` — `states`: işçi başına `(m_k, v_k)`, her biri $w$'nin $k$. parçası boyunda (`np.array_split`). Her işçi kendi parçasını günceller, sonra parçalar birleştirilir (all-gather). → `(w, states)`.

In [ ]:
%%writefile stage_c.py
import numpy as np


def worker_grads(w, X, y, K):
    ...


def allreduce_mean(grads):
    ...


def adam_update(w, g, m, v, t, lr=1e-2, b1=0.9, b2=0.999, eps=1e-8):
    ...


def zero1_step(w, g, states, t, lr=1e-2):
    ...


rng = np.random.default_rng(0)
X, y, w = rng.normal(size=(64, 6)), rng.normal(size=64), rng.normal(size=6)
g = allreduce_mean(worker_grads(w, X, y, 4))
print(g - 2 * X.T @ (X @ w - y) / 64)

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

İki CPU süreci, `gloo`. Süreçler `torch.multiprocessing.start_processes(…, start_method="fork")` ile başlatılır (Windows'ta `fork` yok: WSL ya da Colab kullan).

- `allreduce_mean(t)` — `dist.all_reduce(t, op=SUM)`, sonra `world_size`'a böl (yerinde).
- `worker(rank, world, port, out, steps, manual)` — `init_process_group("gloo", init_method=f"tcp://127.0.0.1:{port}", …)`; `torch.manual_seed(0)` ile `make_model()`; veri `make_data()`, işçi $r$ satırları `r::world` alır; SGD (`lr=0.1`), MSE.
  - `manual=False`: modeli `DistributedDataParallel` ile sar.
  - `manual=True`: DDP yok; `backward`'dan sonra her gradyana `allreduce_mean`.
  - Sonunda parametreleri `out/r{rank}.pt`'ye kaydet, `destroy_process_group()`.
- `run(world, steps, manual)` — boş bir port bul, süreçleri başlat, kaydedilen parametre listelerini döndür.

Dikkat: eşdeğerlik için işçi parçaları **eşit** boyutta olmalı. 64 örneği 3 işçiye bölersen (22, 21, 21) parça ortalamalarının ortalaması tam batch ortalamasına eşit olmaz; `DistributedSampler` bu yüzden veriyi doldurur ya da keser.

In [ ]:
%%writefile stage_d.py
import os
import socket
import tempfile
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP


def make_model():
    return torch.nn.Sequential(torch.nn.Linear(8, 16), torch.nn.Tanh(), torch.nn.Linear(16, 1))


def make_data():
    g = torch.Generator().manual_seed(1)
    return torch.randn(64, 8, generator=g), torch.randn(64, 1, generator=g)


# ---- senin kodun ----

def allreduce_mean(t):
    ...


def worker(rank, world, port, out, steps, manual):
    ...


def run(world, steps=5, manual=False):
    ...


P = run(2)
print(all(torch.equal(a, b) for a, b in zip(P[0], P[1])))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Gerçek kolektiflerle ZeRO-1 (optimizer durumu bölünmüş Adam).

- `flatten(ts)` / `unflatten(flat, like)` — tensör listesini tek düz vektöre ve geri.
- `zero1_worker(rank, world, port, out, steps)`: her adımda
  1. `backward`, gradyanları **tek** düz vektöre topla ve **tek** `all_reduce` (kovalama: çok sayıda küçük çağrı yerine bir büyük çağrı), `world`'e böl;
  2. düz parametre vektörünün kendi parçasını (`world`'e bölünecek şekilde sıfırla doldurulmuş, eşit parçalar) yalnız kendi `m`, `v`'nle Adam ile güncelle;
  3. `dist.all_gather` ile parçaları topla, parametrelere geri yaz.

  Parametreleri ve optimizer durumu eleman sayısını kaydet.
- `run_zero1(world, steps)` → `(parametreler, durum_boyutları)`.

Kontrol tek süreçli `torch.optim.Adam` ile karşılaştırır.

In [ ]:
%%writefile stage_e.py
import os
import socket
import tempfile
import torch
import torch.distributed as dist
import torch.multiprocessing as mp


def make_model():
    torch.manual_seed(0)
    return torch.nn.Sequential(torch.nn.Linear(8, 32), torch.nn.Tanh(), torch.nn.Linear(32, 32), torch.nn.Tanh(), torch.nn.Linear(32, 1))


def make_data():
    g = torch.Generator().manual_seed(1)
    return torch.randn(64, 8, generator=g), torch.randn(64, 1, generator=g)


# ---- senin kodun ----

def flatten(ts):
    ...


def unflatten(flat, like):
    ...


def zero1_worker(rank, world, port, out, steps, lr=1e-2):
    ...


def run_zero1(world, steps=10):
    ...


P, sizes = run_zero1(2)
print(sizes)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Kaynaklar

- [PyTorch Tutorials — Getting Started with Distributed Data Parallel; Writing Distributed Applications](https://docs.pytorch.org/tutorials/intermediate/ddp_tutorial.html) — BSD-3-Clause
- [Li ve ark. — PyTorch Distributed: Experiences on Accelerating Data Parallel Training](https://arxiv.org/abs/2006.15704) — arXiv
- [Rajbhandari ve ark. — ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054) — arXiv
- [Zhao ve ark. — PyTorch FSDP: Experiences on Scaling Fully Sharded Data Parallel](https://arxiv.org/abs/2304.11277) — arXiv
- [Stanford CS336 — Assignment 2 (DDP, sharded optimizer)](https://github.com/stanford-cs336/assignment2-systems) — MIT
- [Gibiansky — Bringing HPC Techniques to Deep Learning (ring all-reduce)](https://andrew.gibiansky.com/blog/machine-learning/baidu-allreduce/) — ©